In [ ]:
# Geneva Cycle & Pedestrian Network Analysis
# -------------------------------------------------
# This script downloads OpenStreetMap data for the Canton of Geneva, then
# computes the share (in linear metres and estimated surface) of bicycle lanes
# and pedestrian‑dedicated paths relative to the overall road network. Finally,
# it renders two minimalist maps à la Action Située.
# -------------------------------------------------
# Usage (from a notebook or terminal):
#   %run geneva_cyclability_analysis.py
# or simply execute the cells step by step if you paste it in a notebook.
# -------------------------------------------------
import osmnx as ox
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt

# -------------------------------------------------
# 1. Fetch the canton boundary and project to a Swiss metric CRS (LV95 / EPSG:2056)
# -------------------------------------------------
area_name = "Canton of Geneva, Switzerland"
print(f"Downloading administrative boundary for: {area_name}")

boundary = ox.geocode_to_gdf(area_name)
if boundary.crs is None:
    boundary.set_crs("EPSG:4326", inplace=True)

boundary = boundary.to_crs("EPSG:2056")
polygon = boundary.geometry.iloc[0]

# -------------------------------------------------
# 2. Retrieve the full network of linear OSM features inside the canton
#    We work with the `all_private` network to keep everything that could be
#    considered navigable (roads, paths, etc.)
# -------------------------------------------------
print("Fetching OSM network … this can take a minute or two ⏳")
G_all = ox.graph_from_polygon(polygon, network_type="all_private", simplify=True)
edges_all = ox.utils_graph.graph_to_gdfs(G_all, nodes=False, edges=True, fill_edge_geometry=True)

# Work in metres ➜ project
edges_all = edges_all.to_crs("EPSG:2056")

# -------------------------------------------------
# 3. Tag bicycle and pedestrian‑exclusive segments
# -------------------------------------------------
cycle_highways = {"cycleway"}
ped_highways   = {"footway", "pedestrian", "path", "steps"}

cycle_edges = edges_all[(edges_all["highway"].isin(cycle_highways)) | edges_all["cycleway"].notna()]
ped_edges   = edges_all[edges_all["highway"].isin(ped_highways)]

# -------------------------------------------------
# 4. Compute linear metrics (metres)
# -------------------------------------------------
print("Computing lengths …")
length_total = edges_all["length"].sum()
length_cycle = cycle_edges["length"].sum()
length_ped   = ped_edges["length"].sum()

# -------------------------------------------------
# 5. Very rough surface estimates using typical widths when explicit width=* is missing
#    (Feel free to update these heuristics with local standards!)
# -------------------------------------------------

def estimate_width(row):
    # if a numeric width tag exists, trust it
    w = row.get("width")
    if pd.notnull(w):
        try:
            return float(str(w).split(";")[0])  # handle "3;3"
        except ValueError:
            pass
    # fall back to heuristic values (in metres)
    if row["highway"] in cycle_highways or pd.notnull(row.get("cycleway")):
        return 1.5  # single‑direction lane
    if row["highway"] in ped_highways:
        return 2.0  # typical sidewalk / footpath
    return 3.5      # generic lane width for the rest of the network

print("Estimating widths and surfaces …")
edges_all["width_est"]  = edges_all.apply(estimate_width, axis=1)
cycle_edges["width_est"] = cycle_edges.apply(estimate_width, axis=1)
ped_edges["width_est"]   = ped_edges.apply(estimate_width, axis=1)

surface_total = (edges_all["length"] * edges_all["width_est"]).sum()
surface_cycle = (cycle_edges["length"] * cycle_edges["width_est"]).sum()
surface_ped   = (ped_edges["length"] * ped_edges["width_est"]).sum()

# -------------------------------------------------
# 6. Summaries
# -------------------------------------------------
length_share_cycle = length_cycle / length_total * 100
length_share_ped   = length_ped   / length_total * 100

surface_share_cycle = surface_cycle / surface_total * 100
surface_share_ped   = surface_ped   / surface_total * 100

print("\n=======================  RESULTS  =======================")
print(f"Total network length       : {length_total:,.0f} m")
print(f" – Cycle‑dedicated length   : {length_cycle:,.0f} m  ({length_share_cycle:.2f} %)")
print(f" – Pedestrian‑dedicated len.: {length_ped:,.0f} m  ({length_share_ped:.2f} %)")
print("--------------------------------------------------------")
print(f"Total network surface       : {surface_total/1e6:,.2f} ha")
print(f" – Cycle‑dedicated surface  : {surface_cycle/1e6:,.2f} ha ({surface_share_cycle:.2f} %)")
print(f" – Pedestrian‑dedicated surf: {surface_ped/1e6:,.2f} ha ({surface_share_ped:.2f} %)")
print("========================================================\n")

# -------------------------------------------------
# 7. Minimalist visualisations (static) – one map per mode
# -------------------------------------------------
print("Drawing minimalist maps …")
plt.rcParams["figure.figsize"] = (8, 8)

# Cycleways map
ax1 = boundary.plot(facecolor="none")
edges_all.plot(ax=ax1, linewidth=0.3, alpha=0.3)
cycle_edges.plot(ax=ax1, linewidth=0.8)
ax1.set_title("Cycleways • Canton de Genève", fontsize=14)
ax1.set_axis_off()
plt.tight_layout()
plt.show()

# Pedestrian paths map
ax2 = boundary.plot(facecolor="none")
edges_all.plot(ax=ax2, linewidth=0.3, alpha=0.3)
ped_edges.plot(ax=ax2, linewidth=0.8)
ax2.set_title("Voies piétonnes • Canton de Genève", fontsize=14)
ax2.set_axis_off()
plt.tight_layout()
plt.show()

print("Done ✅")
